# 05 · Gold — Modelo dimensional

Construye el modelo en estrella sobre `silver.transacciones` para consumo analítico.

```
    dim_fecha ─┐
    dim_canal ─┼── fact_operaciones ──┬─ dim_instrumento
               │                      └─ dim_cliente (SCD Tipo 2)
```

| Tabla | Grano | Notas |
|---|---|---|
| `fact_operaciones` | Una transacción | Une el cliente con la versión vigente en su `fecha_operativa`, y cada operación con su **precio de mercado** (Task 04) |
| `dim_cliente` | Una versión de cliente | **SCD Tipo 2** sobre `segmento_actividad` y `canal_principal` |
| `dim_instrumento` | Un símbolo | Descripción canónica si hay varias |
| `dim_canal` | Un canal | Tipo de canal: APP, WEB, API, OTRO |
| `dim_fecha` | Un día | Días hábiles y feriados |
| `cobertura_cotizaciones` | Un símbolo | % de operaciones con precio de mercado y motivo cuando falta |

Decisiones: DT-007 (segmentación), DT-015 (modelo), DT-016 (SCD2) y DT-020 (imputación de precios) en `docs/TECHNICAL_DECISIONS.md`.

**Prerrequisito:** `setup/05_gold.sql` y los notebooks `02_silver_transacciones` y `04_silver_cotizaciones`.

## Parámetros

In [0]:
dbutils.widgets.text("catalog", "workspace", "Catálogo")
CATALOG = dbutils.widgets.get("catalog")

In [0]:
import json
import os
import sys
from datetime import datetime, timezone

# La lógica vive en src/ para poder testearla sin cluster (ver tests/).
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from pyspark.sql import functions as F

from iol_pipeline.config import PipelineConfig
from iol_pipeline.silver_cotizaciones import IMPUTATION_MAX_BUSINESS_DAYS, coverage_by_symbol, match_market_prices
from iol_pipeline.sources import QUOTE_SOURCES
from iol_pipeline.gold import (
    SEGMENT_THRESHOLDS,
    apply_scd2,
    build_dim_canal,
    build_dim_cliente,
    build_dim_fecha,
    build_dim_instrumento,
    build_fact,
    client_snapshot,
    mask_client_id,
    monthly_cutoffs,
    validate_gold,
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

cfg = PipelineConfig(catalog=CATALOG)
silver_table = cfg.silver_table("transacciones")
tables = {name: cfg.gold_table(name) for name in ["dim_fecha", "dim_canal", "dim_instrumento", "dim_cliente"]}
tables["fact"] = cfg.gold_table("fact_operaciones")
coverage_table = cfg.gold_table("cobertura_cotizaciones")
quotes_table = cfg.silver_table("cotizaciones")
errors_table = cfg.bronze_table("errores_api")

print(f"Origen: {silver_table}")
for name, table in tables.items():
    print(f"{name:16} {table}")

Origen: workspace.silver.transacciones
dim_fecha        workspace.gold.dim_fecha
dim_canal        workspace.gold.dim_canal
dim_instrumento  workspace.gold.dim_instrumento
dim_cliente      workspace.gold.dim_cliente
fact             workspace.gold.fact_operaciones


## Precondiciones

In [0]:
assert spark.catalog.tableExists(silver_table), f"No existe {silver_table}. Ejecutar setup/04_silver.sql"
for table in [*tables.values(), coverage_table]:
    assert spark.catalog.tableExists(table), f"No existe {table}. Ejecutar setup/05_gold.sql"
for table in [quotes_table, errors_table]:
    assert spark.catalog.tableExists(table), f"No existe {table}. Ejecutar setup/06_cotizaciones.sql"

silver = spark.table(silver_table)
silver_rows = silver.count()
assert silver_rows > 0, f"{silver_table} está vacía. Ejecutar 02_silver_transacciones"

bounds = silver.agg(
    F.min("fecha_local").alias("fecha_min"),
    F.min("fecha_operativa").alias("operativa_min"),
    F.max("fecha_operativa").alias("operativa_max"),
).first()
processed_at = datetime.now(timezone.utc).replace(tzinfo=None)
print(f"Filas en silver: {silver_rows} | fechas operativas: {bounds['operativa_min']} a {bounds['operativa_max']}")

Filas en silver: 100000 | fechas operativas: 2026-01-02 a 2026-03-13


## Dimensiones simples
Se reconstruyen completas en cada corrida. Las claves subrogadas son deterministas (`xxhash64` de la clave natural), así la reconstrucción no cambia las claves.

In [0]:
build_dim_fecha(spark, bounds["fecha_min"].replace(day=1), bounds["operativa_max"]).writeTo(tables["dim_fecha"]).overwrite(F.lit(True))
build_dim_canal(silver).writeTo(tables["dim_canal"]).overwrite(F.lit(True))
build_dim_instrumento(silver).writeTo(tables["dim_instrumento"]).overwrite(F.lit(True))

display(spark.createDataFrame(
    [(name, spark.table(tables[name]).count()) for name in ["dim_fecha", "dim_canal", "dim_instrumento"]],
    "tabla STRING, filas LONG",
))

tabla,filas
dim_fecha,72
dim_canal,5
dim_instrumento,1445


In [0]:
display(spark.table(tables["dim_canal"]).orderBy("tipo_canal", "canal"))

canal_key,canal,tipo_canal
3846291822891687862,API,API
9018735379221961643,App Mobile,APP
-5706328406481756524,IOLnet,OTRO
-3052379844113053185,Sitio Web Desktop,WEB
-5146185294737539357,Sitio Web Responsive,WEB


Símbolos con más de una descripción en la fuente: la dimensión toma la más frecuente como canónica.

In [0]:
display(spark.table(tables["dim_instrumento"]).where("descripciones_distintas > 1").select("simbolo", "descripcion", "descripciones_distintas", "tipo_instrumento"))

simbolo,descripcion,descripciones_distintas,tipo_instrumento
AZN,Cedear Astrazeneca Plc,2,CEDEAR
IRS2W,WARRANT INVERSIONES Y REPRESENTACIONES S.A.,2,WARRANT
X15Y6,Letra Tesoro Nacional Ajustable Cer A Descuento15/05/26 $,2,LETRA


## Dimensión de clientes — SCD Tipo 2

El dataset no trae un maestro de clientes, así que los atributos se **derivan del comportamiento** (DT-007): en producción se combinarían con los del CRM usando el mismo mecanismo.

| Atributo versionado | Regla |
|---|---|
| `segmento_actividad` | Operaciones acumuladas: 1 → OCASIONAL · 2-4 → RECURRENTE · 5-19 → FRECUENTE · 20+ → INTENSIVO |
| `canal_principal` | Canal con más operaciones acumuladas (desempate: el más reciente) |

Los cortes mensuales se aplican **en orden**, como cargas sucesivas. Cada carga compara los atributos con la versión vigente (`MERGE`): si cambiaron, cierra la vigente y abre una nueva.

In [0]:
print("Umbrales de segmento:", SEGMENT_THRESHOLDS)
cutoffs = monthly_cutoffs(bounds["operativa_min"], bounds["operativa_max"])
loads = build_dim_cliente(spark, silver, tables["dim_cliente"], cutoffs, processed_at)
display(spark.createDataFrame(loads))

Umbrales de segmento: [(20, 'INTENSIVO'), (5, 'FRECUENTE'), (2, 'RECURRENTE'), (1, 'OCASIONAL')]


con_cambios,corte,fecha_efectiva,nuevos,sin_cambios
0,2026-01-31,2026-01-01,31123,0
4626,2026-02-28,2026-02-01,18139,26497
3190,2026-03-13,2026-03-01,8393,46072


### Demostración con un caso concreto
Un cliente con tres versiones (el id se muestra enmascarado, DT-013).

In [0]:
dim_cliente = spark.table(tables["dim_cliente"])
example_id = (
    dim_cliente.groupBy("id_cliente").agg(F.count("*").alias("versiones"))
    .orderBy(F.desc("versiones"), "id_cliente").first()["id_cliente"]
)
example_mask = mask_client_id(example_id)
print(f"Cliente de ejemplo: {example_mask}")

display(
    dim_cliente.where(F.col("id_cliente") == example_id)
    .select(F.lit(example_mask).alias("cliente"), "segmento_actividad", "canal_principal", "valid_from", "valid_to", "is_current")
    .orderBy("valid_from")
)

Cliente de ejemplo: CLI***B85AC


cliente,segmento_actividad,canal_principal,valid_from,valid_to,is_current
CLI***B85AC,OCASIONAL,App Mobile,2026-01-01,2026-01-31,false
CLI***B85AC,RECURRENTE,App Mobile,2026-02-01,2026-02-28,false
CLI***B85AC,FRECUENTE,App Mobile,2026-03-01,9999-12-31,true


### Idempotencia del SCD2
Re-aplicar el último corte sobre la dimensión ya cargada no debe generar cambios.

In [0]:
last_start, last_cutoff = cutoffs[-1]
rerun = apply_scd2(spark, tables["dim_cliente"], client_snapshot(silver, last_cutoff), last_start, processed_at)
display(spark.createDataFrame([{**rerun, "idempotente": rerun["nuevos"] == 0 and rerun["con_cambios"] == 0}]))

con_cambios,fecha_efectiva,idempotente,nuevos,sin_cambios
0,2026-03-01,true,0,57655


### Distribución
Versiones por cliente y segmento vigente.

In [0]:
display(
    dim_cliente.groupBy("id_cliente").agg(F.count("*").alias("versiones"))
    .groupBy("versiones").agg(F.count("*").alias("clientes")).orderBy("versiones")
)

versiones,clientes
1,50031
2,7432
3,192


In [0]:
display(
    dim_cliente.where("is_current").groupBy("segmento_actividad")
    .agg(F.count("*").alias("clientes")).orderBy(F.desc("clientes"))
)

segmento_actividad,clientes
OCASIONAL,40841
RECURRENTE,14387
FRECUENTE,2257
INTENSIVO,170


## Tabla de hechos
Cada operación recibe el precio de mercado de su `fecha_operativa` (DT-020):

| `tipo_precio` | Regla |
|---|---|
| `EXACTO` | Cierre del mismo día. Una operación de sábado ya está imputada al lunes por su `fecha_operativa` (DT-004) |
| `IMPUTADO` | El símbolo no cotizó ese día: último cierre de hasta 5 días hábiles antes |
| `SIN_COTIZACION` | Sin cierre en esa ventana o sin fuente para el instrumento. La operación se conserva con precio nulo |

In [0]:
market_prices = match_market_prices(
    spark, silver.select("id_transaccion", "simbolo", "fecha_operativa", "precio"), spark.table(quotes_table)
)
build_fact(silver, dim_cliente, processed_at, market_prices).writeTo(tables["fact"]).overwrite(F.lit(True))
fact = spark.table(tables["fact"])
print(f"Filas en fact_operaciones: {fact.count()}")

Filas en fact_operaciones: 100000


### Point-in-time: las operaciones del cliente de ejemplo
Cada operación apunta a la versión vigente en su fecha operativa: las de enero quedan con el segmento de enero, aunque hoy el cliente tenga otro.

In [0]:
display(
    fact.join(dim_cliente, "cliente_sk").where(F.col("id_cliente") == example_id)
    .join(spark.table(tables["dim_fecha"]), "fecha_key")
    .groupBy("nombre_mes", "mes", "segmento_actividad", "canal_principal", "valid_from")
    .agg(F.count("*").alias("operaciones"))
    .orderBy("mes")
    .drop("mes")
)

nombre_mes,segmento_actividad,canal_principal,valid_from,operaciones
enero,OCASIONAL,App Mobile,2026-01-01,1
febrero,RECURRENTE,App Mobile,2026-02-01,3
marzo,FRECUENTE,App Mobile,2026-03-01,1


## Precio de mercado
Distribución por tipo de precio y desvío del precio operado contra el de mercado.

In [0]:
print(f"Ventana de imputación: {IMPUTATION_MAX_BUSINESS_DAYS} días hábiles")
display(
    fact.groupBy("tipo_precio").agg(F.count("*").alias("operaciones"))
    .withColumn("pct", F.round(100 * F.col("operaciones") / silver_rows, 2)).orderBy(F.desc("operaciones"))
)

Ventana de imputación: 5 días hábiles


tipo_precio,operaciones,pct
EXACTO,85158,85.16
SIN_COTIZACION,14822,14.82
IMPUTADO,20,0.02


In [0]:
display(
    fact.where("precio_mercado IS NOT NULL AND dq_status = 'OK'")
    .join(spark.table(tables["dim_instrumento"]).select("instrumento_key", "tipo_instrumento"), "instrumento_key")
    .groupBy("tipo_instrumento", "fuente_precio")
    .agg(F.count("*").alias("operaciones"),
         F.round(F.expr("percentile(desvio_vs_mercado_pct, 0.5)"), 2).alias("desvio_mediana_pct"),
         F.round(F.expr("percentile(abs(desvio_vs_mercado_pct), 0.9)"), 2).alias("desvio_abs_p90_pct"))
    .orderBy(F.desc("operaciones"))
)

tipo_instrumento,fuente_precio,operaciones,desvio_mediana_pct,desvio_abs_p90_pct
CEDEAR,data912,27011,0.09,3.65
CEDEAR,yahoo,18252,0.11,5.12
ACCION,data912,15310,0.15,5.24
BONO,data912,14742,-0.02,1.32
ACCION,yahoo,1102,0.28,6.04
ON,data912,45,0.43,2.45


## Cobertura de cotizaciones por símbolo
Se publica en `gold.cobertura_cotizaciones`. El motivo de los faltantes sale de `bronze.errores_api` (última extracción de cada fuente).

In [0]:
coverage_by_symbol(
    fact, spark.table(tables["dim_instrumento"]), spark.table(errors_table), QUOTE_SOURCES, processed_at
).writeTo(coverage_table).overwrite(F.lit(True))
coverage = spark.table(coverage_table)

display(
    coverage.groupBy("tipo_instrumento", "tipo_liquidacion")
    .agg(F.count("*").alias("simbolos"), F.sum("operaciones").alias("operaciones"),
         F.sum(F.col("ops_precio_exacto") + F.col("ops_precio_imputado")).alias("ops_con_precio"),
         F.sum("ops_precio_imputado").alias("ops_imputadas"))
    .withColumn("pct_ops_con_precio", F.round(100 * F.col("ops_con_precio") / F.col("operaciones"), 1))
    .orderBy(F.desc("operaciones"))
)

tipo_instrumento,tipo_liquidacion,simbolos,operaciones,ops_con_precio,ops_imputadas,pct_ops_con_precio
CEDEAR,ARS,353,47520,44423,14,93.5
ACCION,ARS,66,17099,17088,0,99.9
BONO,ARS,70,13165,10991,0,83.5
CEDEAR,MEP,228,6414,6051,6,94.3
BONO,MEP,38,5524,5275,0,95.5
LETRA,ARS,22,3941,0,0,0.0
OPCION,ARS,299,2160,0,0,0.0
ON,ARS,162,1440,49,0,3.4
ON,MEP,148,1146,0,0,0.0
ACCION,MEP,23,1069,1036,0,96.9


Símbolos con más operaciones sin precio de mercado, y por qué.

In [0]:
display(
    coverage.where("ops_sin_cotizacion > 0")
    .select("simbolo", "tipo_instrumento", "tipo_liquidacion", "operaciones", "ops_sin_cotizacion", "pct_con_precio", "motivo_sin_cotizacion")
    .orderBy(F.desc("ops_sin_cotizacion")).limit(20)
)

simbolo,tipo_instrumento,tipo_liquidacion,operaciones,ops_sin_cotizacion,pct_con_precio,motivo_sin_cotizacion
VIST,CEDEAR,ARS,967,967,0.0,data912: NOT_FOUND
S17A6,LETRA,ARS,694,694,0.0,data912: NOT_FOUND
S27F6,LETRA,ARS,663,663,0.0,data912: NOT_FOUND
S29Y6,LETRA,ARS,398,398,0.0,data912: NOT_FOUND
S16M6,LETRA,ARS,389,389,0.0,data912: NOT_FOUND
UNH,CEDEAR,ARS,298,298,0.0,data912: NOT_FOUND
S30A6,LETRA,ARS,291,291,0.0,data912: NOT_FOUND
TZX26,BONO,ARS,287,287,0.0,data912: NOT_FOUND
D30A6,BONO,ARS,249,249,0.0,data912: NOT_FOUND
AAL,CEDEAR,ARS,238,238,0.0,data912: NOT_FOUND


In [0]:
totals = coverage.agg(
    F.sum("operaciones").alias("ops"), F.sum("ops_precio_exacto").alias("exacto"),
    F.sum("ops_precio_imputado").alias("imputado"), F.sum("ops_sin_cotizacion").alias("sin"),
    F.count("*").alias("simbolos"), F.sum((F.col("pct_con_precio") > 0).cast("int")).alias("simbolos_con_precio"),
).first()
coverage_summary = {
    "pct_ops_con_precio": round(100 * (totals["exacto"] + totals["imputado"]) / totals["ops"], 2),
    "pct_ops_imputadas": round(100 * totals["imputado"] / totals["ops"], 2),
    "simbolos_con_precio": f"{totals['simbolos_con_precio']}/{totals['simbolos']}",
}
print(coverage_summary)

{'pct_ops_con_precio': 85.18, 'pct_ops_imputadas': 0.02, 'simbolos_con_precio': '645/1445'}


## Controles de integridad
- El hecho conserva todas las filas de Silver.
- Toda clave del hecho existe en su dimensión.
- Cada cliente tiene exactamente una versión vigente y sus vigencias no se superponen.

In [0]:
result = validate_gold(spark, tables, silver_rows)
display(spark.createDataFrame([{**result.to_dict(), "orphan_keys": json.dumps(result.orphan_keys)}]))

clients_without_single_current,fact_rows,message,orphan_keys,overlapping_versions,silver_rows,status
0,100000,"100000 operaciones modeladas, integridad referencial y SCD2 OK","{""fecha_key"": 0, ""instrumento_key"": 0, ""canal_key"": 0, ""cliente_sk"": 0}",0,100000,OK


## Ejemplo de consulta sobre el modelo
Operaciones por segmento (al momento de operar) y tipo de instrumento.

In [0]:
display(spark.sql(f"""
    SELECT c.segmento_actividad,
           i.tipo_instrumento,
           COUNT(*)                                   AS operaciones,
           COUNT(DISTINCT c.id_cliente)               AS clientes
    FROM {tables['fact']} f
    JOIN {tables['dim_cliente']} c     ON f.cliente_sk = c.cliente_sk
    JOIN {tables['dim_instrumento']} i ON f.instrumento_key = i.instrumento_key
    GROUP BY c.segmento_actividad, i.tipo_instrumento
    ORDER BY c.segmento_actividad, operaciones DESC
"""))

segmento_actividad,tipo_instrumento,operaciones,clientes
FRECUENTE,CEDEAR,8888,1922
FRECUENTE,ACCION,2866,1025
FRECUENTE,BONO,1385,594
FRECUENTE,OPCION,517,124
FRECUENTE,ON,353,158
FRECUENTE,LETRA,304,189
FRECUENTE,WARRANT,10,9
INTENSIVO,CEDEAR,1969,141
INTENSIVO,ACCION,1494,104
INTENSIVO,OPCION,929,22


## Resultado
Se devuelve al orquestador como JSON.

In [0]:
summary = {**result.to_dict(), "cargas_scd2": loads, "cobertura": coverage_summary}
print(json.dumps(summary, indent=2, default=str))
dbutils.notebook.exit(json.dumps(summary, default=str))